In [ ]:
import random
import pandas as pd
import re
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import ElasticNet, Ridge
from xgboost import XGBRegressor
import tensorflow as tf
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor

In [ ]:
df_0=pd.read_csv("/content/drive/MyDrive/df_0.csv") #for validation
df_1=pd.read_csv("/content/drive/MyDrive/df_1.csv")
df_2=pd.read_csv("/content/drive/MyDrive/df_2.csv")
df_3=pd.read_csv("/content/drive/MyDrive/df_3.csv")
df_4=pd.read_csv("/content/drive/MyDrive/df_4.csv")

In [ ]:
dfs = [df_1, df_2, df_3, df_4]

# Adding noise

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import find_peaks
from scipy.interpolate import interp1d

def add_peak_shifts_to_spectra(
    df,
    shift_prob=1.0, 
    shift_std=4.0,
    window_half_cm=100.0,
    min_peak_height=0.99,  
    min_peak_distance_cm=10.0,
    random_seed=52
):
    """
    Applies random local peak shifts to infrared spectra to simulate instrumental drift 
    and enhance model robustness against peak position variability.

    This augmentation technique mimics realistic spectral variations caused by factors 
    such as instrument calibration drift, temperature fluctuations, or sample preparation 
    differences. Shifts are applied locally around detected absorption peaks using 
    interpolation within a defined spectral window.

    Parameters
    ----------
    df : pd.DataFrame
        Input spectral data where rows represent individual samples and columns represent 
        wavenumbers (cm⁻¹).
    shift_prob : float, default=1.0
        Probability [0, 1] of applying a shift to each detected absorption peak.
    shift_std : float, default=4.0
        Standard deviation of the Gaussian-distributed shift magnitude (in cm⁻¹).
    window_half_cm : float, default=100.0
        Half-width of the local interpolation window centered at each peak (in cm⁻¹). 
        The full window spans ±window_half_cm around the peak position.
    min_peak_height : float, default=0.99
        Maximum transmittance threshold for peak detection. Peaks are identified as 
        local minima where transmittance ≤ min_peak_height (i.e., absorption features 
        with intensity ≥ 1 - min_peak_height).
    min_peak_distance_cm : float, default=10.0
        Minimum separation between adjacent peaks (in cm⁻¹) to avoid duplicate detection.
    random_seed : int or None, default=52
        Seed for random number generator to ensure reproducibility of augmentations.

    Returns
    -------
    pd.DataFrame
        Augmented spectral data with identical shape and indexing as input DataFrame. 
        Peak positions are perturbed while preserving overall spectral morphology.

    Notes
    -----
    - Peak detection operates on transmittance spectra (minima correspond to absorption peaks).
    - Shifts are applied via linear interpolation within localized windows to maintain 
      spectral continuity.
    """

    # Ensure wavenumber axis is numeric and sorted in descending order (IR convention)
    x = np.array(df.columns, dtype=float)
    if not np.all(x[:-1] > x[1:]):
        # Sort columns if not already in descending order
        sort_idx = np.argsort(-x)
        x = x[sort_idx]
        df = df.iloc[:, sort_idx]

    # Calculate spectral resolution (assumes uniform wavenumber spacing)
    dx = np.abs(x[1] - x[0])
    min_distance_points = int(np.ceil(min_peak_distance_cm / dx))

    result = []

    for idx in df.index:
        y = df.loc[idx].values.copy()
        y_augmented = y.copy()

        # Detect absorption peaks (local minima in transmittance spectra)
        # Using inverted signal (-y) to leverage find_peaks which detects maxima
        peaks, _ = find_peaks(
            -y,
            height=-min_peak_height,      # Convert transmittance threshold to inverted scale
            distance=min_distance_points  # Enforce minimum peak separation
        )

        # Apply random shifts to detected peaks based on probability
        for peak_idx in peaks:
            if np.random.rand() > shift_prob:
                continue  # Skip this peak according to probability

            # Generate random shift magnitude (Gaussian distribution)
            delta_cm = np.random.normal(0, shift_std)
            delta_cm = np.clip(delta_cm, -window_half_cm / 2, window_half_cm / 2)
            delta_points = int(np.round(delta_cm / dx))

            # Define local window boundaries around the peak
            left = max(0, peak_idx - int(window_half_cm / dx))
            right = min(len(x) - 1, peak_idx + int(window_half_cm / dx))

            x_window = x[left:right + 1]
            y_window = y[left:right + 1]

            # Create shifted wavenumber axis for interpolation
            x_shifted = x_window + delta_cm

            # Interpolate shifted spectrum back onto original wavenumber grid
            interp_func = interp1d(
                x_shifted, 
                y_window,
                kind='linear',
                bounds_error=False,
                fill_value=1.0  # Default to 100% transmittance (baseline) outside interpolation range
            )
            y_shifted_window = interp_func(x_window)

            # Apply smooth blending at window edges to prevent discontinuities
            win_len = len(y_shifted_window)
            blend_width = max(1, win_len // 8)  # Minimum 1-point transition zone
            blend = np.ones(win_len)
            
            if blend_width * 2 < win_len:
                # Linear ramp for smooth transition at boundaries
                blend[:blend_width] = np.linspace(0, 1, blend_width)
                blend[-blend_width:] = np.linspace(1, 0, blend_width)
            # Else: too narrow window for blending – apply full shift

            # Blend original and shifted spectra within the window
            y_augmented[left:right + 1] = (
                blend * y_shifted_window + (1 - blend) * y_window
            )

        result.append(y_augmented)

    # Return DataFrame with identical structure to input
    return pd.DataFrame(result, index=df.index, columns=df.columns)

In [ ]:
def preprocess_spectra(df):
  part1 = df.iloc[:, 56:1779:3]
  part1=add_peak_shifts_to_spectra(part1)

  part2 = df.iloc[:, -4:]
  new_data = pd.concat([part1, part2], axis=1)
  return new_data

In [ ]:
preproc_spectra=[]
for df in dfs:
  df_i = preprocess_spectra(df)
  preproc_spectra.append(df_i)

In [ ]:
X_2=preproc_spectra[0].iloc[:, :575]

In [ ]:
def corr_mat(X):
  correlation_matrix = X.corr()  
  corr_no_diag = correlation_matrix.mask(np.eye(len(correlation_matrix), dtype=bool))

  mask = np.triu(np.ones_like(corr_no_diag, dtype=bool))


  sns.heatmap(
      corr_no_diag,
      mask=mask,
      cmap='coolwarm',
      vmin=-1, vmax=1,
      center=0,
      square=True,
      linewidths=0,  
      annot=False )

  plt.show()
  np.fill_diagonal(correlation_matrix.values, 0)


  mean_corr = correlation_matrix.values.mean()
  print(f"Mean correlaion: {mean_corr:.3f}")

In [ ]:
correlation_matrix = X_2.corr() 
corr_no_diag = correlation_matrix.mask(np.eye(len(correlation_matrix), dtype=bool))

mask = np.triu(np.ones_like(corr_no_diag, dtype=bool))


sns.heatmap(
      corr_no_diag,
      mask=mask,
      cmap='coolwarm',
      vmin=-1, vmax=1,
      center=0,
      square=True,
      linewidths=0,  
      annot=False 
  )

plt.show()
np.fill_diagonal(correlation_matrix.values, 0)
mean_corr = correlation_matrix.values.mean()
print(f"Mean correlaion: {mean_corr:.3f}")

In [ ]:
threshold = 0.7

In [ ]:
weak_correlated_features = []
for col in correlation_matrix.columns:
    if (corr_no_diag[col].abs() < threshold).any():
        weak_correlated_features.append(col)


In [ ]:
weak_counts = (corr_no_diag.abs() < threshold).sum(axis=0)

weak_counts_sorted = weak_counts.sort_values(ascending=False)

In [ ]:
top_n = int(len(weak_counts) * 0.1773)  
top_weak_features = weak_counts_sorted.index[:top_n].tolist()

In [ ]:
X_y_weak_correlation=[]

for preproc_spectrum in preproc_spectra:
  X = preproc_spectrum.iloc[:, :575]
  Y=preproc_spectrum.iloc[:, 575:]
  X_weak_correlation=X[top_weak_features]
  data = pd.concat([X_weak_correlation, Y], axis=1)
  X_y_weak_correlation.append(data)

# Random Forest

In [ ]:
model = RandomForestRegressor(random_state=42)

r2_all = []
mse_all = []

for i in range(4):
    test_df = X_y_weak_correlation[i]
    train_df = pd.concat([df for j, df in enumerate(X_y_weak_correlation) if j != i], ignore_index=True)

    X_train = train_df.iloc[:, :-4]  
    y_train = train_df.iloc[:, -4:]

    X_test = test_df.iloc[:, :-4]
    y_test = test_df.iloc[:, -4:]


    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred, multioutput='raw_values')
    mse = mean_squared_error(y_test, y_pred, multioutput='raw_values')

    r2_all.append(r2)
    mse_all.append(mse)

r2_all = np.array(r2_all)
mse_all = np.array(mse_all)

Y = train_df.iloc[:, -4:].values 
for i in range(Y.shape[1]):
    print(f"Output {i+1}:")
    print(f"  R² scores:  {r2_all[:, i]} | Mean R²:  {np.mean(r2_all[:, i]):.4f}")
    print(f"  MSE scores: {mse_all[:, i]} | Mean MSE: {np.mean(mse_all[:, i]):.4f}")
    print()

In [ ]:
train_full_df = pd.concat([df for df in X_y_weak_correlation])
X_train_full = train_full_df.iloc[:, :-4]  
y_train_full = train_full_df.iloc[:, -4:]

In [ ]:
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train_full, y_train_full)

RandomForestRegressor(max_depth=15, max_features='sqrt', min_samples_leaf=3,
                      min_samples_split=5, n_estimators=300, oob_score=True,
                      random_state=42)

In [ ]:
df_0_preproc=preprocess_spectra(df_0)
X_val = df_0_preproc.iloc[:, :575]
Y_val=df_0_preproc.iloc[:, 575:]
X_weak_correlation_val=X_val[top_weak_features]


In [ ]:
y_pred_val= rf.predict(X_weak_correlation_val)

r2 = r2_score(Y_val, y_pred_val, multioutput='raw_values')
mse = mean_squared_error(Y_val, y_pred_val, multioutput='raw_values')
print('R2', r2)
print('mse', mse)

# XGBoost

In [ ]:
r2_all = []
mse_all = []

for i in range(4):
    test_df = X_y_weak_correlation[i]

    train_df = pd.concat([df for j, df in enumerate(X_y_weak_correlation) if j != i],
                         ignore_index=True)

    X_train = train_df.iloc[:, :-4]
    y_train = train_df.iloc[:, -4:]

    X_test = test_df.iloc[:, :-4]
    y_test = test_df.iloc[:, -4:]

    y_train_sub = y_train.iloc[:, 1:]

    base_model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42
    )
    model = MultiOutputRegressor(base_model)
    model.fit(X_train, y_train_sub)

    y_pred_ = model.predict(X_test)
    
    r2 = r2_score(y_test, y_pred, multioutput='raw_values')
    mse = mean_squared_error(y_test, y_pred, multioutput='raw_values')

    r2_all.append(r2)
    mse_all.append(mse)

r2_all = np.array(r2_all)
mse_all = np.array(mse_all)

Y = train_df.iloc[:, -4:].values
for i in range(Y.shape[1]):
    print(f"Output {i+1}:")
    print(f"  R² scores:  {r2_all[:, i]} | Mean R²:  {np.mean(r2_all[:, i]):.4f}")
    print(f"  MSE scores: {mse_all[:, i]} | Mean MSE: {np.mean(mse_all[:, i]):.4f}")
    print()


In [ ]:
train_full_df = pd.concat([df for df in X_y_weak_correlation])
X_train_full = train_full_df.iloc[:, :-4] 
y_train_full = train_full_df.iloc[:, -4:]

In [ ]:
sigma = 0.001  

X_train_full = X_train_full + np.random.normal(0, sigma, size=X_train_full.shape)

In [ ]:
base_model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42)
model = MultiOutputRegressor(base_model)
model.fit(X_train_full, y_train_full)

MultiOutputRegressor(estimator=XGBRegressor(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=0.5, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric=None,
                                            feature_types=None,
                                            feature_weights=None, gamma=None,
                                            grow_policy=None,
                                            importance_type=None,
                                            interaction_constraints=None,
                                            learning_rate=None, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=None,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=None, n_jobs=None,
                                            num_parallel_tree=None, ...))

In [ ]:
df_0_preproc=preprocess_spectra(df_0)
X_val = df_0_preproc.iloc[:, :575]
Y_val=df_0_preproc.iloc[:, 575:]
X_weak_correlation_val=X_val[top_weak_features]

X_weak_correlation_val = X_weak_correlation_val + np.random.normal(0, sigma, size=X_weak_correlation_val.shape)

In [ ]:
y_pred_val= model.predict(X_weak_correlation_val)

r2 = r2_score(Y_val, y_pred_val, multioutput='raw_values')
mse = mean_squared_error(Y_val, y_pred_val, multioutput='raw_values')
print('R2', r2)
print('mse', mse)

# CNN

In [ ]:
train_full_df = pd.concat([df for df in preproc_spectra])
X_train_full = train_full_df.iloc[:, :-4]
X_train_full = X_train_full + np.random.normal(0, sigma, size=X_train_full.shape)
y_train_full = train_full_df.iloc[:, -4:]

In [ ]:
np.random.seed(42)
X = X_train_full.values.astype(np.float32)  
y = y_train_full.values.astype(np.float32)
y = y / y.sum(axis=1, keepdims=True)  # normalize to distribution

# TARGET VARIABLE PREPARATION

# Protect against zeros (to avoid log(0) in future losses)
epsilon = 1e-6
y = np.clip(y, epsilon, 1 - 3 * epsilon)  
y = y / y.sum(axis=1, keepdims=True)      

# SPECTRA NORMALIZATION
scaler = StandardScaler()
X_flat = X.reshape(-1, 575)
X_scaled = scaler.fit_transform(X_flat).astype(np.float32)
X_3d = X_scaled.reshape(-1, 575, 1)  # (n_samples, 575, 1)


# TRAIN/VALIDATION SPLIT

X_train, X_val, y_train, y_val = train_test_split(
    X_3d, y, test_size=0.15, random_state=42
)

# MODEL BUILDING

def build_1d_cnn(input_length=575, n_outputs=4):
    model = tf.keras.Sequential([
        # Block 1
        tf.keras.layers.Conv1D(64, kernel_size=7, activation='relu', input_shape=(input_length, 1)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling1D(pool_size=2),

        # Block 2
        tf.keras.layers.Conv1D(128, kernel_size=5, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling1D(pool_size=2),

        # Block 3
        tf.keras.layers.Conv1D(256, kernel_size=3, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling1D(pool_size=2),

        # Block 4
        tf.keras.layers.Conv1D(256, kernel_size=3, activation='relu'),
        tf.keras.layers.BatchNormalization(),

        # Reduce to feature vector
        tf.keras.layers.GlobalAveragePooling1D(),

        # Fully connected part
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.3),

        # Output
        tf.keras.layers.Dense(n_outputs),
        tf.keras.layers.Softmax()  # ← ensures sum = 1
    ])
    return model

model = build_1d_cnn(input_length=575, n_outputs=4)


# COMPILATION AND CALLBACKS

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',  # works with soft labels!
    metrics=['mae']
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, verbose=1)
]

# TRAINING

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)


# EVALUATION

print("\nEvaluation on validation set:")
y_pred = model.predict(X_val)

mae_per_component = mean_absolute_error(y_val, y_pred, multioutput='raw_values')
print(f"MAE per component: {mae_per_component}")
print(f"Average MAE: {mae_per_component.mean():.4f}")


In [ ]:
y_pred_test = model.predict(X_val)

In [ ]:
r2_per_component  = r2_score(y_val, y_pred_test, multioutput='raw_values')  
mse_per_component = mean_squared_error(y_val, y_pred_test, multioutput='raw_values')

print("R2:", r2_per_component)
print("MSE:", mse_per_component)

In [ ]:
df_0_preproc=preprocess_spectra(df_0)
X_val = df_0_preproc.iloc[:, :575]
X_val = X_val + np.random.normal(0, sigma, size=X_val.shape)
Y_val=df_0_preproc.iloc[:, 575:]

In [ ]:
X_val_scaled = scaler.transform(X_val)  
X_val_3d = X_val_scaled.reshape(-1, 575, 1)

In [ ]:
y_pred = model.predict(X_val_3d)

In [ ]:
y_pred = model.predict(X_val)

In [ ]:
mae_per_component = mean_absolute_error(Y_val, y_pred, multioutput='raw_values')
mae_avg = mae_per_component.mean()

rmse_per_component = np.sqrt(mean_squared_error(Y_val, y_pred, multioutput='raw_values'))
rmse_avg = rmse_per_component.mean()

print("MAE по компонентам:", mae_per_component)
print("Средний MAE:", mae_avg)
print("RMSE по компонентам:", rmse_per_component)

In [ ]:
r2_per_component  = r2_score(Y_val, y_pred, multioutput='raw_values') 
mse_per_component = mean_squared_error(Y_val, y_pred, multioutput='raw_values')

print("R2:", r2_per_component)
print("MSE:", mse_per_component)